In [4]:
!pip install openmeteo-requests
!pip install requests-cache retry-requests

  Using cached openmeteo_requests-1.7.3-py3-none-any.whl.metadata (11 kB)
  Using cached niquests-3.15.2-py3-none-any.whl.metadata (16 kB)
  Using cached openmeteo_sdk-1.21.2-py3-none-any.whl.metadata (935 bytes)
  Using cached urllib3_future-2.14.901-py3-none-any.whl.metadata (16 kB)
  Using cached wassima-2.0.1-py3-none-any.whl.metadata (3.7 kB)
  Using cached flatbuffers-25.2.10-py2.py3-none-any.whl.metadata (875 bytes)
  Using cached h11-0.16.0-py3-none-any.whl.metadata (8.3 kB)
  Using cached jh2-5.0.9-cp37-abi3-win_amd64.whl.metadata (4.0 kB)
  Using cached qh3-1.5.4-cp37-abi3-win_amd64.whl.metadata (4.8 kB)
Using cached openmeteo_requests-1.7.3-py3-none-any.whl (7.0 kB)
Using cached niquests-3.15.2-py3-none-any.whl (167 kB)
Using cached openmeteo_sdk-1.21.2-py3-none-any.whl (17 kB)
Using cached flatbuffers-25.2.10-py2.py3-none-any.whl (30 kB)
Using cached urllib3_future-2.14.901-py3-none-any.whl (675 kB)
Using cached wassima-2.0.1-py3-none-any.whl (145 kB)
Using cached h11-0.16.

In [5]:
import openmeteo_requests

import pandas as pd
import requests_cache
from retry_requests import retry

In [11]:
# Setup the Open-Meteo API client with cache and retry on error
cache_session = requests_cache.CachedSession('.cache', expire_after = -1)
retry_session = retry(cache_session, retries = 5, backoff_factor = 0.2)
openmeteo = openmeteo_requests.Client(session = retry_session)

# Make sure all required weather variables are listed here
# The order of variables in hourly or daily is important to assign them correctly below
url = "https://archive-api.open-meteo.com/v1/archive"
params = {
	"latitude": 35.6768601,
	"longitude": 139.7638947,
	"start_date": "2022-01-18",
	"end_date": "2025-10-02",
	"hourly": ["temperature_2m", "relative_humidity_2m", "precipitation", "rain", "snowfall", "pressure_msl", "cloud_cover", "cloud_cover_low", "cloud_cover_high", "cloud_cover_mid", "wind_speed_10m", "wind_direction_10m", "wind_speed_100m", "wind_direction_100m", "weather_code"],
	# "bounding_box": "-90,-180,90,180",
}
responses = openmeteo.weather_api(url, params=params)

# Process bounding box locations
for response in responses:
	print(f"\nCoordinates: {response.Latitude()}°N {response.Longitude()}°E")
	print(f"Elevation: {response.Elevation()} m asl")
	print(f"Timezone difference to GMT+0: {response.UtcOffsetSeconds()}s")
	
	# Process hourly data. The order of variables needs to be the same as requested.
	hourly = response.Hourly()
	hourly_temperature_2m = hourly.Variables(0).ValuesAsNumpy()
	hourly_relative_humidity_2m = hourly.Variables(1).ValuesAsNumpy()
	hourly_precipitation = hourly.Variables(2).ValuesAsNumpy()
	hourly_rain = hourly.Variables(3).ValuesAsNumpy()
	hourly_snowfall = hourly.Variables(4).ValuesAsNumpy()
	hourly_pressure_msl = hourly.Variables(5).ValuesAsNumpy()
	hourly_cloud_cover = hourly.Variables(6).ValuesAsNumpy()
	hourly_cloud_cover_low = hourly.Variables(7).ValuesAsNumpy()
	hourly_cloud_cover_high = hourly.Variables(8).ValuesAsNumpy()
	hourly_cloud_cover_mid = hourly.Variables(9).ValuesAsNumpy()
	hourly_wind_speed_10m = hourly.Variables(10).ValuesAsNumpy()
	hourly_wind_direction_10m = hourly.Variables(11).ValuesAsNumpy()
	hourly_wind_speed_100m = hourly.Variables(12).ValuesAsNumpy()
	hourly_wind_direction_100m = hourly.Variables(13).ValuesAsNumpy()
	hourly_weather_code = hourly.Variables(14).ValuesAsNumpy()
	
	hourly_data = {"date": pd.date_range(
		start = pd.to_datetime(hourly.Time(), unit = "s", utc = True),
		end = pd.to_datetime(hourly.TimeEnd(), unit = "s", utc = True),
		freq = pd.Timedelta(seconds = hourly.Interval()),
		inclusive = "left"
	)}
	
	hourly_data["temperature_2m"] = hourly_temperature_2m
	hourly_data["relative_humidity_2m"] = hourly_relative_humidity_2m
	hourly_data["precipitation"] = hourly_precipitation
	hourly_data["rain"] = hourly_rain
	hourly_data["snowfall"] = hourly_snowfall
	hourly_data["pressure_msl"] = hourly_pressure_msl
	hourly_data["cloud_cover"] = hourly_cloud_cover
	hourly_data["cloud_cover_low"] = hourly_cloud_cover_low
	hourly_data["cloud_cover_high"] = hourly_cloud_cover_high
	hourly_data["cloud_cover_mid"] = hourly_cloud_cover_mid
	hourly_data["wind_speed_10m"] = hourly_wind_speed_10m
	hourly_data["wind_direction_10m"] = hourly_wind_direction_10m
	hourly_data["wind_speed_100m"] = hourly_wind_speed_100m
	hourly_data["wind_direction_100m"] = hourly_wind_direction_100m
	hourly_data["weather_code"] = hourly_weather_code
	
	hourly_dataframe = pd.DataFrame(data = hourly_data)
	hourly_dataframe


Coordinates: 35.6766242980957°N 139.80694580078125°E
Elevation: 27.0 m asl
Timezone difference to GMT+0: 0s


In [12]:
hourly_dataframe.to_csv("data/weather.csv", index = False)

In [13]:
hourly_dataframe['temperature_2m']

0         4.031000
1         5.381000
2         5.931000
3         6.631000
4         6.881000
           ...    
32491    19.281000
32492    18.880999
32493    18.580999
32494    19.130999
32495    20.480999
Name: temperature_2m, Length: 32496, dtype: float32